In [8]:
# notebooks/eda.ipynb
# Exploratory Data Analysis for All Curated ACIC Datasets

import pandas as pd
import numpy as np
import os
from glob import glob
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
# --- Paths ---
input_dir = '../data/inputs/curated_data/'
output_dir = '../data/outputs/eda/'
os.makedirs(output_dir, exist_ok=True)

In [10]:
# --- Dataset list ---
data_files = sorted(glob(os.path.join(input_dir, '*.csv')))
print(f"Found {len(data_files)} datasets")

Found 18 datasets


In [11]:
# --- Programmatically identify numeric and categorical columns ---
def identify_column_types(df, exclude_cols=None):
    if exclude_cols is None:
        exclude_cols = []

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

    numeric_cols = [c for c in numeric_cols if c not in exclude_cols]
    cat_cols = [c for c in cat_cols if c not in exclude_cols]

    return numeric_cols, cat_cols

# --- Numeric summary ---
def eda_numeric_summary(df, numeric_cols):
    summary_list = []
    for col in numeric_cols:
        s = df[col].describe()
        s['skew'] = df[col].skew()
        s['kurtosis'] = df[col].kurtosis()
        summary_df = pd.DataFrame(s).T
        summary_df.index = [col]
        summary_list.append(summary_df)
    return summary_list

# --- Categorical summary ---
def eda_categorical_summary(df, cat_cols):
    summary_list = []
    for col in cat_cols:
        counts = df[col].value_counts(dropna=False)
        percents = df[col].value_counts(normalize=True, dropna=False) * 100
        summary_df = pd.DataFrame({'count': counts, 'percent': percents})
        summary_df.index.name = col
        summary_list.append(summary_df)
    return summary_list

In [12]:
# --- Loop over datasets ---
for file_path in data_files:
    dataset_id = os.path.basename(file_path).split('.')[0].split('_')[-1]
    print(f"\n=== Processing dataset {dataset_id} ===")

    data = pd.read_csv(file_path)

    # --- Identify columns ---
    exclude_cols = ['ID']  # always exclude IDs
    numeric_cols, cat_cols = identify_column_types(data, exclude_cols=exclude_cols)

    # --- Numeric summaries ---
    num_summary_list = eda_numeric_summary(data, numeric_cols)
    # --- Save numeric summaries ---
    for summary_df in num_summary_list:
        col = summary_df.index[0]  # column name is the index
        summary_df.to_csv(os.path.join(output_dir, f'numeric_summary_{col}_{dataset_id}.csv'))

    # --- Categorical summaries ---
    cat_summary_list = eda_categorical_summary(data, cat_cols)
    for summary_df in cat_summary_list:
        # The column name is stored as the index name in the DataFrame
        col = summary_df.index.name
        summary_df.to_csv(os.path.join(output_dir, f'categorical_summary_{col}_{dataset_id}.csv'))

    # --- Plots ---
    # 1) Boxplot of y by treatment
    plt.figure(figsize=(6,4))
    sns.boxplot(x='z', y='y', data=data)
    plt.title(f'Outcome y by Treatment - Dataset {dataset_id}')
    plt.savefig(os.path.join(output_dir, f'y_by_treatment_{dataset_id}.png'))
    plt.close()

    # 2) Histograms/density plots of numeric covariates
    for col in numeric_cols:
        if col == 'y':
            continue  # skip outcome if needed
        if pd.api.types.is_numeric_dtype(data[col]):
            plt.figure(figsize=(6,3))
            sns.histplot(data[col], kde=True)
            plt.title(f'Distribution of {col} - Dataset {dataset_id}')
            plt.savefig(os.path.join(output_dir, f'{col}_dist_{dataset_id}.png'))
            plt.close()

    # 3) Correlation heatmap for numeric covariates
    plt.figure(figsize=(12,10))
    numeric_cols_for_corr = [c for c in numeric_cols if pd.api.types.is_numeric_dtype(data[c])]
    corr_matrix = data[numeric_cols_for_corr].corr()
    sns.heatmap(corr_matrix, cmap='coolwarm', center=0)
    plt.title(f'Correlation Heatmap - Dataset {dataset_id}')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'corr_heatmap_{dataset_id}.png'))
    plt.close()

    # 4) Boxplots of selected covariates by treatment (first 6 for clarity)
    numeric_cols_for_boxplots = [c for c in numeric_cols if pd.api.types.is_numeric_dtype(data[c])]
    for col in numeric_cols_for_boxplots[:6]:  # first 6 numeric columns
        plt.figure(figsize=(6,4))
        sns.boxplot(x='z', y=col, data=data)
        plt.title(f'{col} by Treatment - Dataset {dataset_id}')
        plt.savefig(os.path.join(output_dir, f'{col}_by_treatment_{dataset_id}.png'))
        plt.close()

print("\nEDA completed for all datasets. Outputs saved in data/outputs/eda/")


=== Processing dataset 1 ===

=== Processing dataset 10 ===

=== Processing dataset 11 ===

=== Processing dataset 12 ===

=== Processing dataset 13 ===

=== Processing dataset 14 ===

=== Processing dataset 15 ===

=== Processing dataset 16 ===

=== Processing dataset 17 ===

=== Processing dataset 18 ===

=== Processing dataset 2 ===

=== Processing dataset 3 ===

=== Processing dataset 4 ===

=== Processing dataset 5 ===

=== Processing dataset 6 ===

=== Processing dataset 7 ===

=== Processing dataset 8 ===

=== Processing dataset 9 ===

EDA completed for all datasets. Outputs saved in data/outputs/eda/
